# inat-creature-crop
Jev finds up to `max_subjects` animals per photo: whole animal (green), head (pink), feet (cyan). Each one is rotated (moth: head up; butterfly from above: head up; otherwise feet down or head to the side) and square-cropped.

Settings and questions are in `prompt.json`, logic in `boxes.py`. Put `TYPESAFE_API_KEY=...` in `inat-creature-crop/.env`.

In [ ]:
import importlib
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd

import boxes
importlib.reload(boxes)  # pick up prompt.json / boxes.py edits without restarting

HERE = Path(boxes.__file__).parent
OUT = HERE / "output"; OUT.mkdir(exist_ok=True)
photos = sorted(p for p in (HERE / "photos").iterdir() if p.suffix.lower() in boxes.PHOTO_EXTS)
print(len(photos), "photos")

In [ ]:
rows = []
for path in photos:
    try:
        r = boxes.process(path)
    except Exception as e:
        display(Markdown(f"**{path.name}**: {e}")); continue
    p = r["preview"].copy(); p.thumbnail((450, 450))
    display(Markdown(f"### {path.name}  ({r['count']} found)")); display(p)
    for j, s in enumerate(r["subjects"], 1):
        name = f"{path.stem}_crop.jpg" if j == 1 else f"{path.stem}_crop{j}.jpg"
        s["crop"].save(OUT / name, quality=92)
        c = s["crop"].copy(); c.thumbnail((300, 300))
        display(Markdown(f"{j}. {s['kind']}, {s['view']} view, {s['rule']}, rotated `{s['angle']}°`, conf `{s['confidence']}`")); display(c)
        rows.append({"photo": path.name, "crop": name, **{k: s[k] for k in ("kind", "view", "rule", "angle", "confidence")}})
pd.DataFrame(rows)